# Day 6 · Afternoon exercises (Exercise B)

**13:30–15:00 · then everyone back in the room at 15:00 for Peer review B**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Keep the folders `data/` and `models/` next to this notebook.

**If Stan does not compile on your laptop**, continue on UCloud or on a group member's laptop.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| B1 | A pilot study: within every target versus overall | 10 min |
| B2 | Two logistic models, and a fair overall comparison | 15 min |
| B3 | The whole class: does feedback help, and does the model fit? | 20 min |
| B4 | **Challenge, no template:** the Monty Hall problem by simulation | 25 min |
| — | Exit response and your group's question set | 15 min |

**Roles this block:** rotate them from the morning.

**The data.** B1–B2 use `data/pilot-study.csv`: a **synthetic** pilot study of the feedback game at another university, 480 attempts, in which the targets were not assigned evenly between the two conditions. B3 uses the class's own Day 1 data (`data/timing-class.csv`). A **hit** is an attempt within 0.5 s of the target.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
theme_set(theme_minimal(base_size = 14) + theme(legend.position = "top"))
DATA <- "data"; MODELS <- "models"     # folders next to this notebook
condition_colours <- c("without feedback" = "navy", "with feedback" = "tomato")

In [ ]:
pilot <- read_csv(file.path(DATA, "pilot-study.csv"), show_col_types = FALSE) |>   # attempt, feedback, target_seconds, hit
  mutate(x = target_seconds - 10,
         condition = factor(if_else(feedback == 1, "with feedback", "without feedback"),
                            levels = c("without feedback", "with feedback")))
pilot |> count(condition, target_seconds)

# The class's Day 1 data: both games, 56 people, 30 attempts per game
d_all <- read_csv(file.path(DATA, "timing-class.csv"), show_col_types = FALSE) |>
  mutate(hit       = as.integer(abs(elapsed_seconds - target_seconds) <= 0.5),   # 1 = within 0.5 s of the target
         feedback  = as.integer(feedback_condition == "feedback"),              # 1 = with feedback
         x         = target_seconds - 10,                                       # centred target
         condition = factor(if_else(feedback == 1, "with feedback", "without feedback"),
                            levels = c("without feedback", "with feedback")))
nrow(d_all)

## B1 (10 min): a pilot study, within every target versus overall

*Slides "Does feedback help? A pilot study", "Within every target versus overall" and "Why it happens".*

**Tasks.**

1. Make one table with hits out of attempts and the share of hits for each target and condition, and one for each condition over all targets. Plot both in one figure, with a legend.
2. Does feedback help within each target? Overall? Explain the difference with the mix of targets in each condition (the share of each condition's attempts at each target).
3. Reweight: what would the overall share with feedback be if its attempts had the same mix of targets as the attempts without feedback?

In [ ]:
# Task 1
# by_target <- pilot |> group_by(group = paste(target_seconds, "s"), condition) |>
#   summarise(hits = sum(hit), attempts = n(), .groups = "drop")
# overall <- pilot |> group_by(group = "all targets", condition) |>
#   summarise(hits = sum(hit), attempts = n(), .groups = "drop")
# table_pilot <- bind_rows(by_target, overall) |> mutate(share = hits / attempts,
#   group = factor(group, levels = c("5 s", "10 s", "15 s", "all targets")))

# Task 2: the mix of targets
# mix <- pilot |> count(condition, target_seconds) |> group_by(condition) |> mutate(weight = n / sum(n)) |> ungroup()

**Your answer:** write a few sentences here.

## B2 (15 min): two logistic models, and a fair overall comparison

*Slides "Two logistic models, two answers", "Which comparison answers the question?" and "A fair overall comparison".* The same Stan program as this morning (`models/logistic.stan`), with two design matrices:

- Model 1: X has one column, feedback (`prior_sd = 1`);
- Model 2: X has two columns, x and feedback (`prior_sd = c(0.5, 1)`), and X_new has the six cases: targets 5, 10, 15 s without feedback (rows 1–3) and with feedback (rows 4–6).

**Tasks.**

1. Fit both models. Report the 90% interval of the feedback coefficient and its odds ratio in each, and P(β_f > 0 | y).
2. Which comparison answers "does feedback help a player at a given target?", and why? What in the design of the pilot made the two answers differ?
3. From Model 2, average P(hit) over an **equal** mix of the three targets, with and without feedback. Give the difference with a 90% interval and P(difference > 0 | y).

In [ ]:
# mod <- cmdstan_model(file.path(MODELS, "logistic.stan"))
# data1 <- list(N = nrow(pilot), K = 1, X = cbind(feedback = pilot$feedback), y = pilot$hit,
#               prior_sd = 1, M = 0, X_new = matrix(0, nrow = 0, ncol = 1))
# cases <- expand_grid(feedback = c(0, 1), target_seconds = c(5, 10, 15))
# data2 <- list(N = nrow(pilot), K = 2, X = cbind(x = pilot$x, feedback = pilot$feedback), y = pilot$hit,
#               prior_sd = c(0.5, 1), M = nrow(cases), X_new = cbind(cases$target_seconds - 10, cases$feedback))
# fit1 <- mod$sample(data = data1, chains = 4, seed = 2026, refresh = 0)
# fit2 <- mod$sample(data = data2, chains = 4, seed = 2026, refresh = 0)

# Task 3
# draws2 <- fit2$draws(format = "df") |> as_tibble()
# p_without <- (draws2$`p_new[1]` + draws2$`p_new[2]` + draws2$`p_new[3]`) / 3
# p_with    <- (draws2$`p_new[4]` + draws2$`p_new[5]` + draws2$`p_new[6]`) / 3

**Your answer:** write a few sentences here.

## B3 (20 min): the whole class, does feedback help, and does the model fit?

*Slides "Which comparison answers the question?", "A predictive check for yes/no data" and "The modelling workflow".* Now the class's own data: 56 people, every person tried every target in both games.

**Tasks.**

1. Make the table of hits by target and condition, and by condition over all targets. Is there a paradox here? Why (look at the design)?
2. Fit Model 2 (x and feedback) to all 3,360 attempts. Report the odds ratio for feedback with a 90% interval, and P(hit) at each target with and without feedback.
3. **Check the model by person.** The model treats all 3,360 attempts as independent, with the same P(hit) for everybody at a given target and condition. Statistic: the SD across people of their number of hits (out of 60). Compare the observed SD with the replicated SDs, using 200 posterior draws (code below). What does the check show, and which model from this course would address it?
4. Feedback was always the **second** game. What else, apart from feedback, differs between the two conditions? What would a better design do?

In [ ]:
# Task 1: as in B1, with d_all instead of pilot

# Task 2
# cases <- expand_grid(feedback = c(0, 1), target_seconds = c(5, 10, 15))
# data_class <- list(N = nrow(d_all), K = 2, X = cbind(x = d_all$x, feedback = d_all$feedback), y = d_all$hit,
#                    prior_sd = c(0.5, 1), M = nrow(cases), X_new = cbind(cases$target_seconds - 10, cases$feedback))
# fit_class <- mod$sample(data = data_class, chains = 4, seed = 2026, refresh = 0)

# Task 3
# draws_class <- fit_class$draws(format = "df") |> as_tibble()
# observed_sd <- d_all |> group_by(participant) |> summarise(hits = sum(hit)) |> summarise(sd_hits = sd(hits))
# replicated_sd <- draws_class |> slice_sample(n = 200) |>
#   select(.draw, alpha, b1 = `beta[1]`, b2 = `beta[2]`) |>
#   cross_join(d_all |> select(participant, x, feedback)) |>
#   mutate(y_rep = rbinom(n(), 1, plogis(alpha + b1 * x + b2 * feedback))) |>
#   group_by(.draw, participant) |> summarise(hits = sum(y_rep), .groups = "drop") |>
#   group_by(.draw) |> summarise(sd_hits = sd(hits))

**Your answer:** write a few sentences here.

## B4 (25 min): challenge, no template — the Monty Hall problem by simulation

*Slides "The Monty Hall problem", "Bayes' rule, with the host's action as the data", "Why switching wins" and "Change the host's rule".* The slides derive the answers with Bayes' rule. Now check them by simulation, in R, with no Stan and no starter code. Plan the steps in your group first.

**Tasks.**

1. **The usual host.** Simulate S = 10,000 games: the car is behind a random door, you always pick door 1, and the host opens a door with a goat among doors 2 and 3 (at random if both have goats). Estimate P(win | stay) and P(win | switch), each with its Monte Carlo standard error (*Day 3*). Do they agree with 1/3 and 2/3?
2. **The forgetful host.** Now the host opens door 2 or 3 at random, and sometimes shows the car. Keep only the games in which he shows a goat. How many of the 10,000 games are kept, and why about that many? Estimate P(win | switch) in the kept games.
3. **Your simulation as Bayes' rule.** In task 1, among the games in which the host opened door 3, what share had the car behind door 1, 2 and 3? Compare with the posterior column on the slide.
4. **100 doors.** Now there are n = 100 doors with one car. You pick door 1. The host, who knows where the car is, opens **98** of the other doors, all with goats, so that one other door stays closed; you may switch to it. Write a simulation that works for any n, and estimate P(win | stay) and P(win | switch) for n = 3, 10 and 100. Is switching better, and by how much? Explain the result for n = 100 in one sentence, and give the exact P(win | switch) for n doors.
5. *Optional.* With 100 doors, the host opens only **one** goat door, and you switch to one of the other 98 closed doors at random. Is switching still better than staying? By how much?

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **One row per game:** `games <- tibble(game = 1:S, car = sample(1:3, S, replace = TRUE))`.
- **A random choice in every row:** first draw a column of random choices for all games, for example `coin = sample(c(2, 3), S, replace = TRUE)`, then use it only where the host is free to choose.
- **The usual host's door:** `case_when(car == 2 ~ 3, car == 3 ~ 2, TRUE ~ coin)`: he must avoid the car.
- **The door you switch to:** with three doors, the closed door that is neither 1 nor the opened door is `5 - opened`.
- **Wins:** `win_stay = car == 1`; `mean(win_stay)` is the estimate, and its Monte Carlo standard error is `sqrt(p * (1 - p) / S)`.
- **Keeping only some games:** `filter(car != opened)`.
- **Shares by door:** `filter(opened == 3) |> count(car) |> mutate(share = n / sum(n))`.
- **n doors:** the host must leave the car closed. So if the car is not behind door 1, the door left closed is the car's door; if it is behind door 1, he leaves one of doors 2, …, n closed at random. Write that as a function `play(n)` that returns the door left closed for one game, and repeat it with a `for` loop (or `replicate()`).
- **Exact answer:** when does switching lose? How likely is that?
- **Optional task 5:** `setdiff(2:n, c(car, opened))` lists the doors you could switch to; `sample(x, 1)` picks one (careful: if `x` has length 1, `sample(x, 1)` draws from `1:x`, so use `x[sample.int(length(x), 1)]`).

</details>

In [ ]:
# B4: your code

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: What is Simpson's paradox? Using the pilot study and the class data, explain when the overall comparison and the comparison within targets agree, and when they do not.

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 15:00 for Peer review B.** Save this notebook (Ctrl/Cmd + S).

Thank you for taking part in the course. The final assignment is due on **13 December 2026 at 23:55** on Moodle.